In [ ]:
import sys
sys.path.append("..")
sys.path.append("../api")

from api.AirbnbRemoteData import AirbnbRemoteData

berlin_listings_url = "https://data.insideairbnb.com/germany/be/berlin/2025-09-23/visualisations/listings.csv"
munich_listing_url = "https://data.insideairbnb.com/germany/bv/munich/2025-09-27/visualisations/listings.csv"

airbnb_fetcher = AirbnbRemoteData(berlin_listings_url, munich_listing_url)
airbnb_fetcher.fetch_data()

In [ ]:
columns_df_a = airbnb_fetcher.get_berlin_data_frame_columns()
print(columns_df_a)

In [ ]:
columns_df_b = airbnb_fetcher.get_munich_data_frame_columns()
print(columns_df_b)

In [ ]:
merged_df = airbnb_fetcher.get_combined_data()
if merged_df is not None:
    print(f"Successfully merged!")
    display(merged_df.head(500))
else:
    print("Merge failed. Check if the CSVs share a common column name (like 'id' or 'listing_id').")

In [ ]:
# the columns need to keep for analysis:price 
columns_to_keep = [
    'price',
    'room_type',
    'neighbourhood','city',
    'minimum_nights',
    'number_of_reviews','reviews_per_month','availability_365'
]
print(columns_to_keep)

In [ ]:
merged_df = merged_df[columns_to_keep]
merged_df

In [ ]:
## Cleaning nas
df1=merged_df
df1.isnull().sum()
(df1.isnull().sum() / len(df1)) * 100
df2 = df1.dropna(subset=["price"]) # droping according to price
(df2.isnull().sum() / len(df2)) * 100

#df2.shape

## Dealing with duplicates ################
df2.duplicated().sum()
df3 = df2.drop_duplicates()
df3.duplicated().sum()

#df3.shape

## plotting price ###################
import matplotlib.pyplot as plt
df3["price"].plot(kind="hist", bins=50)
#df3[df3["price"] < 1000]["price"].plot(kind="hist", bins=50) 
#Message berlin_team_1

In [ ]:
df3[df3["price"] < 1000]["price"].plot(kind="hist", bins=50)

In [ ]:
df3["revenue"] = df3["minimum_nights"]* df3["price"]
#df3['revenue'] = df3['minimum_nights'].astype(int) * df3['price'].astype(float)
df3["revenue"] = df3["revenue"].round(2)

In [ ]:
df_berlin = df3[(df3['city'] == 'Berlin') & (df3['price'] < 1000)].copy()

berlin_revenue = df_berlin.groupby(
    ['neighbourhood', 'room_type'], as_index=False)['revenue'].sum()

berlin_revenue = berlin_revenue.sort_values(by='revenue', ascending=False).reset_index(drop=True)
berlin_revenue["revenue"] = berlin_revenue["revenue"].round(2)
berlin_revenue

In [ ]:
df_berlin = df3[(df3['city'] == 'Berlin') & (df3['price'] < 1000)].copy()

berlin_revenue = df_berlin.groupby(['neighbourhood', 'room_type'], as_index=False)['revenue'].mean()

berlin_revenue = berlin_revenue.sort_values(by='revenue', ascending=False)
berlin_revenue = berlin_revenue.reset_index(drop=True)
berlin_revenue["revenue"] = berlin_revenue["revenue"].round(2)
berlin_revenue

In [ ]:
df_munich = df3[(df3['city'] == 'Munich') & (df3['price'] < 1000)].copy()

munich_revenue = df_munich.groupby(['neighbourhood', 'room_type'], as_index=False)['revenue'].mean()

munich_revenue = munich_revenue.sort_values(by='revenue', ascending=False)
munich_revenue = munich_revenue.reset_index(drop=True)
munich_revenue["revenue"] = munich_revenue["revenue"].round(2)
munich_revenue

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

room_type_summary = berlin_revenue[berlin_revenue['room_type'] != 'Hotel room'].groupby('room_type', as_index=False)['revenue'].sum()

room_type_summary = room_type_summary.sort_values(by='revenue', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=room_type_summary, x='room_type', y='revenue', palette='viridis')

plt.title('Total Revenue by Room Type in Berlin', fontsize=15)
plt.xlabel('Room Type', fontsize=12)
plt.ylabel('Total Revenue ($)', fontsize=12)

plt.show()

In [ ]:
room_type_summary = munich_revenue[munich_revenue['room_type']!='Hotel room'].groupby('room_type', as_index=False)['revenue'].sum()

room_type_summary = room_type_summary.sort_values(by='revenue', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=room_type_summary, x='room_type', y='revenue', palette='viridis')

plt.title('Total Revenue by Room Type in Munich', fontsize=15)
plt.xlabel('Room Type', fontsize=12)
plt.ylabel('Total Revenue ($)', fontsize=12)

plt.show()

In [ ]:
room_type_summary = berlin_revenue[berlin_revenue['room_type']!='Hotel room'].head(5).groupby('neighbourhood', as_index=False)['revenue'].sum()

room_type_summary = room_type_summary.sort_values(by='revenue', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=room_type_summary, x='neighbourhood', y='revenue', palette='viridis')

plt.title('Total Revenue by Room Type in Berlin', fontsize=15)
plt.xlabel('Neighbourhood', fontsize=12)
plt.ylabel('Total Revenue ($)', fontsize=12)

plt.show()

In [ ]:
df_berlin = df3[
    (df3['city'] == 'Berlin') & 
    (df3['room_type'] == 'Entire home/apt') &
    (df3['price'] < 1000)].copy()

berlin_revenue = df_berlin.groupby(['neighbourhood', 'room_type'], as_index=False)['revenue'].mean()

berlin_revenue = berlin_revenue.sort_values(by='revenue', ascending=False).reset_index(drop=True)
berlin_revenue["revenue"] = berlin_revenue["revenue"].round(2)
berlin_revenue

In [ ]:
df_berlin = df3[
    (df3['city'] == 'Berlin') & 
    (df3['room_type'] == 'Private room') &
    (df3['price'] < 1000)].copy()

berlin_revenue = df_berlin.groupby(['neighbourhood', 'room_type'], as_index=False)['revenue'].mean()

berlin_revenue = berlin_revenue.sort_values(by='revenue', ascending=False).reset_index(drop=True)
berlin_revenue["revenue"] = berlin_revenue["revenue"].round(2)
berlin_revenue

In [ ]:
df_berlin = df3[
    (df3['city'] == 'Berlin') & 
    (df3['room_type'] == 'Shared room') &
    (df3['price'] < 1000)].copy()

berlin_revenue = df_berlin.groupby(['neighbourhood', 'room_type'], as_index=False)['revenue'].mean()

berlin_revenue = berlin_revenue.sort_values(by='revenue', ascending=False).reset_index(drop=True)
berlin_revenue["revenue"] = berlin_revenue["revenue"].round(2)
berlin_revenue

In [ ]:
df_berlin = df3[
    (df3['city'] == 'Berlin') & 
    (df3['room_type'] == 'Hotel room') &
    (df3['price'] < 1000)].copy()

berlin_revenue = df_berlin.groupby(['neighbourhood', 'room_type'], as_index=False)['revenue'].mean()

berlin_revenue = berlin_revenue.sort_values(by='revenue', ascending=False).reset_index(drop=True)
berlin_revenue["revenue"] = berlin_revenue["revenue"].round(2)
berlin_revenue

In [ ]:
df_munich = df3[
    (df3['city'] == 'Munich') & 
    (df3['room_type'] == 'Entire home/apt') &
    (df3['price'] < 1000)].copy()

munich_revenue = df_munich.groupby(['neighbourhood', 'room_type'], as_index=False)['revenue'].mean()

munich_revenue = munich_revenue.sort_values(by='revenue', ascending=False).reset_index(drop=True)
munich_revenue["revenue"] = munich_revenue["revenue"].round(2)
munich_revenue

In [ ]:
df_munich = df3[
    (df3['city'] == 'Munich') & 
    (df3['room_type'] == 'Private room') &
    (df3['price'] < 1000)].copy()

munich_revenue = df_munich.groupby(['neighbourhood', 'room_type'], as_index=False)['revenue'].mean()

munich_revenue = munich_revenue.sort_values(by='revenue', ascending=False).reset_index(drop=True)
munich_revenue["revenue"] = munich_revenue["revenue"].round(2)
munich_revenue

In [ ]:
df_munich = df3[
    (df3['city'] == 'Munich') & 
    (df3['room_type'] == 'Shared room') &
    (df3['price'] < 1000)].copy()

munich_revenue = df_munich.groupby(['neighbourhood', 'room_type'], as_index=False)['revenue'].mean()

munich_revenue = munich_revenue.sort_values(by='revenue', ascending=False).reset_index(drop=True)
munich_revenue["revenue"] = munich_revenue["revenue"].round(2)
munich_revenue

In [ ]:
df_munich = df3[
    (df3['city'] == 'Munich') & 
    (df3['room_type'] == 'Hotel room') &
    (df3['price'] < 1000)].copy()

munich_revenue = df_munich.groupby(['neighbourhood', 'room_type'], as_index=False)['revenue'].mean()

munich_revenue = munich_revenue.sort_values(by='revenue', ascending=False).reset_index(drop=True)
munich_revenue["revenue"] = munich_revenue["revenue"].round(2)
munich_revenue